In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

def visualize_vog_data(file_path):
    """
    Analyzes a single VOG eye-tracking data file and visualizes 3 core architectural views.
    (Includes Multi-encoding Self-healing & Dynamic Anti-Saccade Target Inversion)
    """
    path_obj = Path(file_path)
    if not path_obj.exists():
        print(f"[Error] File not found: {path_obj.resolve()}")
        return False

    # 1. Dynamic Metadata Extraction for Data Traceability
    group_name = path_obj.parent.parent.name if len(path_obj.parents) >= 2 else "Unknown_Group"
    session_id = path_obj.parent.name
    task_name = path_obj.stem.replace("PD VOG -_", "").replace("PD VOG -", "").strip()

    try:
        header_idx = -1
        header_columns = []
        raw_lines = []

        encodings_to_try = ['utf-16', 'utf-16le', 'utf-8-sig', 'cp949']

        for enc in encodings_to_try:
            try:
                with open(path_obj, 'r', encoding=enc, errors='replace') as f:
                    lines = f.readlines()

                for i, line in enumerate(lines):
                    line_clean = line.replace('\x00', '').lower()
                    if 'lh' in line_clean and 'rh' in line_clean and 'target' in line_clean:
                        header_idx = i
                        raw_lines = lines
                        header_columns = [col.replace('\x00', '').strip() for col in line.split(',')]
                        break

                if header_idx != -1:
                    break
            except UnicodeError:
                continue

        if header_idx == -1:
            print(f"[Skip] Missing core headers (LH, RH, Target) in {task_name}.")
            return False

        parsed_data = []
        for line in raw_lines[header_idx + 1:]:
            line_clean = line.replace('\x00', '').strip()
            if not line_clean:
                continue

            row_values = [val.strip() for val in line_clean.split(',')]

            # Padding & Truncation for Robustness
            if len(row_values) < len(header_columns):
                row_values.extend([''] * (len(header_columns) - len(row_values)))
            elif len(row_values) > len(header_columns):
                row_values = row_values[:len(header_columns)]

            parsed_data.append(row_values)

        df = pd.DataFrame(parsed_data, columns=header_columns)
        df = df.apply(pd.to_numeric, errors='coerce')
        df = df.dropna(how='all').reset_index(drop=True)

    except Exception as e:
        print(f"[Error] Fatal parsing error: {e}")
        return False

    # 2. Dynamic Feature Routing
    time_col = next((col for col in df.columns if 'time' in str(col).lower() or str(col).lower() == 't'), None)
    if not time_col:
        time_col = df.columns[0]
    time_sec = df[time_col]

    target_v_col = next((c for c in df.columns if 'targetv' in str(c).lower() or 'target_v' in str(c).lower()), None)
    target_h_col = next((c for c in df.columns if 'targeth' in str(c).lower() or 'target_h' in str(c).lower()), None)

    target_col = None
    if target_v_col and df[target_v_col].abs().sum() > 0:
        target_col = target_v_col
    elif target_h_col:
        target_col = target_h_col
    elif target_v_col:
        target_col = target_v_col

    if not target_col:
        print(f"[Error] Target column detection failed.")
        return False

    direction_str = "Vertical" if 'v' in str(target_col).lower() else "Horizontal"

    search_l = 'lv' if direction_str == "Vertical" else 'lh'
    search_r = 'rv' if direction_str == "Vertical" else 'rh'

    eye_col_l = next((c for c in df.columns if str(c).lower() == search_l), None)
    eye_col_r = next((c for c in df.columns if str(c).lower() == search_r), None)

    if not eye_col_l or not eye_col_r:
        print(f"[Error] Missing required eye tracking columns ({search_l}, {search_r}).")
        return False

    # =========================================================================
    # [ARCHITECTURAL UPDATE] Dynamic Target Inversion for Anti-Saccade Tasks
    # =========================================================================
    """Changed"""
    is_anti = 'anti' in task_name.lower()

    """Changed"""
    if is_anti:
        # Invert the ground truth: Patient must look in the opposite direction
        df['Expected_Target'] = -df[target_col]
    else:
        df['Expected_Target'] = df[target_col]

    # Feature Engineering: Calculate Tracking Error based on EXPECTED target
    """Changed"""
    df['Error_L'] = df[eye_col_l] - df['Expected_Target']
    df['Error_R'] = df[eye_col_r] - df['Expected_Target']

    # 3. Multi-tier Visualization
    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

    title_str = f"[{group_name}] Session: {session_id} \n Task: {task_name} ({direction_str} Analysis)"
    fig.suptitle(title_str, fontsize=18, fontweight='bold')

    # View 1: Raw Waveform with Expected Anti-Target overlay
    axes[0].plot(time_sec, df[target_col], label=f'Target Stimulus ({target_col})', color='red', linestyle='--', linewidth=2)

    """Changed"""
    if is_anti:
        axes[0].plot(time_sec, df['Expected_Target'], label='Expected Eye Position (Anti)', color='magenta', linestyle=':', linewidth=2)

    axes[0].plot(time_sec, df[eye_col_l], label=f'Left Eye ({eye_col_l})', color='blue', alpha=0.7)
    axes[0].plot(time_sec, df[eye_col_r], label=f'Right Eye ({eye_col_r})', color='green', alpha=0.7)
    axes[0].set_title(f'1. Raw Waveform: Target vs Eye Movement', fontsize=14)
    axes[0].set_ylabel('Position (Degrees)')
    axes[0].legend(loc='upper right')
    axes[0].grid(True, linestyle=':', alpha=0.6)

    # View 2: Tracking Error Waveform
    axes[1].axhline(0, color='red', linestyle='--', linewidth=1)

    """Changed"""
    axes[1].plot(time_sec, df['Error_L'], label=f'Error Left', color='purple', alpha=0.8)
    axes[1].plot(time_sec, df['Error_R'], label=f'Error Right', color='orange', alpha=0.8)
    axes[1].set_title('2. Derived Feature: Eye Tracking Error (Deviation from Expected)', fontsize=14)
    axes[1].set_ylabel('Error Distance')
    axes[1].legend(loc='upper right')
    axes[1].grid(True, linestyle=':', alpha=0.6)

    # View 3: Orthogonal Variance (Cross-Axis Noise)
    noise_col_l = 'LH' if direction_str == "Vertical" else 'LV'
    noise_col_r = 'RH' if direction_str == "Vertical" else 'RV'

    actual_noise_col_l = next((c for c in df.columns if str(c).lower() == noise_col_l.lower()), None)
    actual_noise_col_r = next((c for c in df.columns if str(c).lower() == noise_col_r.lower()), None)

    if actual_noise_col_l and actual_noise_col_r:
        axes[2].plot(time_sec, df[actual_noise_col_l], label=f'Left Eye ({actual_noise_col_l})', color='gray', alpha=0.7)
        axes[2].plot(time_sec, df[actual_noise_col_r], label=f'Right Eye ({actual_noise_col_r})', color='brown', alpha=0.7)
        axes[2].set_title(f'3. Outlier/Noise Monitoring (Orthogonal Cross-Axis Variance)', fontsize=14)
        axes[2].set_xlabel('Time (sec)', fontsize=12)
        axes[2].set_ylabel('Position')
        axes[2].legend(loc='upper right')
        axes[2].grid(True, linestyle=':', alpha=0.6)
    else:
        axes[2].set_title(f'3. Outlier/Noise Monitoring (Data Not Available for {noise_col_l}, {noise_col_r})', fontsize=14)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

    # Explicit Garbage Collection to prevent OOM during batch processing
    plt.close(fig)

    return True

def visualize_directory_recursively(base_dir_path):
    """
    Recursively scans and visualizes all VOG CSV files in the target directory.
    """
    base_path = Path(base_dir_path)

    if not base_path.exists() or not base_path.is_dir():
        print(f"[Error] Invalid directory: {base_path.resolve()}")
        return

    csv_files = list(base_path.rglob('*.csv'))

    if not csv_files:
        print(f"[Info] No CSV files found in '{base_path.name}'.")
        return

    print("=" * 70)
    print(f"🚀 Initializing VOG Data Batch Pipeline")
    print(f"- Target Directory: {base_path.resolve()}")
    print(f"- Total Files Discovered: {len(csv_files)}")
    print("=" * 70)

    success_count = 0
    for i, file_path in enumerate(csv_files, 1):
        print(f"\n▶ [{i}/{len(csv_files)}] Processing: {file_path.name}")

        is_success = visualize_vog_data(file_path)
        if is_success:
            success_count += 1

    print("\n" + "=" * 70)
    print(f"✅ Batch Processing Complete! (Success: {success_count} / Total: {len(csv_files)})")
    print("=" * 70)

if __name__ == "__main__":
    # Point this to your project's data directory
    target_directory = "data/sample_csv"
    visualize_directory_recursively(target_directory)